# Module 5 — Inverse Kinematics
## Unit 7 — Verifying and Connecting to Perception
### Lesson 7.1 — Verifying a Solution with Forward Kinematics

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Demonstration (§7)
Solve → verify → accept/reject by the FK residual.


In [ ]:
import numpy as np

def fk_two_link(t1, t2, L1, L2):
    return np.array([L1*np.cos(t1)+L2*np.cos(t1+t2), L1*np.sin(t1)+L2*np.sin(t1+t2)])

def jacobian_2link(t1, t2, L1, L2):
    s1,s12=np.sin(t1),np.sin(t1+t2); c1,c12=np.cos(t1),np.cos(t1+t2)
    return np.array([[-L1*s1-L2*s12,-L2*s12],[L1*c1+L2*c12,L2*c12]])

def ik_2link_closed(x, y, L1, L2):
    c2=(x*x+y*y-L1*L1-L2*L2)/(2*L1*L2)
    if c2<-1-1e-9 or c2>1+1e-9: return []
    c2=max(-1.0,min(1.0,c2)); sols=[]
    for sign in (+1.0,-1.0):
        s2=sign*np.sqrt(max(0.0,1-c2*c2)); t2=np.arctan2(s2,c2)
        t1=np.arctan2(y,x)-np.arctan2(L2*np.sin(t2),L1+L2*np.cos(t2))
        sols.append((t1,t2))
        if abs(s2)<1e-6: break
    return sols

def reachable(x, y, L1, L2, tol=1e-9):
    r=np.hypot(x,y); return abs(L1-L2)-tol<=r<=L1+L2+tol

def ik_numerical(target, theta0, L1, L2, alpha=1.0, tol=1e-6, max_iter=100):
    theta=np.array(theta0,float); target=np.array(target,float)
    for it in range(max_iter):
        e=target-fk_two_link(theta[0],theta[1],L1,L2)
        if np.linalg.norm(e)<tol: return theta
        J=jacobian_2link(theta[0],theta[1],L1,L2)
        theta=theta+alpha*np.linalg.pinv(J)@e
    return theta

def norm180(a): return (a+180.0)%360.0-180.0

def verify(theta_c, target, L1, L2, tol=1e-3):
    p=fk_two_link(theta_c[0],theta_c[1],L1,L2)
    res=np.linalg.norm(np.array(target)-p)
    return res<tol, res

L1,L2=0.4,0.3; target=(0.5,0.2)
good=ik_2link_closed(*target,L1,L2)[0]
print("good candidate:", verify(good,target,L1,L2))
bad=(good[0]+0.1, good[1]-0.1)
print("perturbed candidate:", verify(bad,target,L1,L2))


## Coding Exercise (§8)
Accept the closed-form solution; reject a stalled solve and a reflected pose.


In [ ]:
# YOUR CODE HERE
ok,res=verify(ik_2link_closed(*target,L1,L2)[0], target, L1,L2); assert ok and res<1e-9
stalled=ik_numerical(target, np.radians([10,20]), L1,L2, max_iter=3)   # too few iters
ok2,res2=verify(stalled,target,L1,L2); assert (not ok2) and res2>1e-3
reflected=(good[0], -good[1])
ok3,_=verify(reflected,target,L1,L2); assert not ok3
print("verify accept/reject OK")


## Check your work


In [ ]:
accepted,res=verify(ik_2link_closed(*target,L1,L2)[0],target,L1,L2)
assert accepted
print("All checks passed.")
